# Cop & Thief — Results Analysis Notebook

Guidelines §9.2 (Results Analysis Notebook). This notebook (1) loads the **live bonus-series §9.2 report** and charts it, and (2) runs a small **parameter-sensitivity sweep** with the offline heuristic agents and charts Cop win-rate vs vision radius. The full study lives in [`docs/EXPERIMENTS.md`](../docs/EXPERIMENTS.md); the headless sweep script is [`parameter_sweep.py`](parameter_sweep.py).

Run it with:

```bash
uv sync --extra analysis
uv run jupyter lab notebooks/results_analysis.ipynb
```

No LLM key, cloud, or credentials are needed — everything here is offline and reproducible from the per-series seeds.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Resolve the repo root whether the kernel starts in repo/ or repo/notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))  # for `parameter_sweep`
print("repo root:", ROOT)

## 1. The live bonus series (§9.2 report)

Loads the most recent persisted bonus report (`results/side-*/bonus_report.json`); falls back to the committed sample if none is present. The bars show each sub-game's role scores, and the title carries the series winner and team totals.

In [ ]:
reports = sorted((ROOT / "results").glob("side-*/bonus_report.json"))
src = reports[-1] if reports else (ROOT / "docs/examples/sample_report.json")
report = json.loads(src.read_text())
print("source:", src.relative_to(ROOT))

sg = report["sub_games"]
idx = [g["index"] for g in sg]
cop = [g["cop_score"] for g in sg]
thief = [g["thief_score"] for g in sg]

fig, ax = plt.subplots(figsize=(7, 3))
w = 0.4
ax.bar([i - w / 2 for i in idx], cop, w, label="cop_score")
ax.bar([i + w / 2 for i in idx], thief, w, label="thief_score")
ax.set_xlabel("sub-game")
ax.set_ylabel("score")
ax.set_xticks(idx)
ax.set_title(
    f"Per-sub-game scores — winner: {report.get('series_winner', '?')} "
    f"| totals: {report['totals_by_group']}"
)
ax.legend()
plt.tight_layout()
plt.show()

report["totals_by_group"], report.get("series_winner"), report.get("mutual_agreement")

## 2. Parameter sensitivity — vision radius on the 5×5 board

Reuses `parameter_sweep.run_batch` (offline heuristic agents) to play several seeded series at each vision radius and plot the Cop win-rate. `n_series` is kept small for a fast notebook; `docs/EXPERIMENTS.md` reports the full-seed study.

In [ ]:
from parameter_sweep import run_batch  # noqa: E402  (path set above)

from cop_thief.shared.config import load_config

base = load_config()
radii = [1, 2, 3]
rates, avgs = [], []
for r in radii:
    cop_wins, total, avg_moves = run_batch(base, [5, 5], r, n_series=6)
    rates.append(100 * cop_wins / total)
    avgs.append(avg_moves)
    print(f"r{r}: cop win-rate {cop_wins}/{total} = {cop_wins / total:.0%}, avg moves {avg_moves:.1f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(radii, rates, "o-")
ax.set_xlabel("vision_radius")
ax.set_ylabel("Cop win-rate (%)")
ax.set_xticks(radii)
ax.set_ylim(0, 100)
ax.set_title("5×5 — Cop win-rate vs vision radius (heuristic agents)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Interpretation

- **Bonus series:** the loaded §9.2 report is internally consistent — team totals equal the summed role scores, and `series_winner` matches the per-sub-game wins.
- **Vision sweep:** at the **radius-1** local default the fog yields a genuine contest (~50–55% Cop); raising vision toward **radius 2** swings the tiny 5×5 board to the Cop, as a strong pursuer corners the evader. This is the expected pursuit-evasion structure and is **symmetric across the bonus role-swap**, so the inter-team match stays fair.
- Full analysis (more seeds, barrier ablation, search-vs-heuristic): [`docs/EXPERIMENTS.md`](../docs/EXPERIMENTS.md).